Let's check if the catalog and schema exists

In [0]:
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
volume_name = dbutils.widgets.get("volume_name")

volume = f"{catalog}.{schema}.{volume_name}"
volume_path = f"{catalog}/{schema}/{volume_name}"

assert catalog, f"catalog:{catalog} is empty"
assert schema, f"schema:{schema} is empty"
assert volume_name, f"volume:{volume_path} is empty"


Let's assert if earthquake connection and try to get the EarthQuake data via API.

In [0]:
from databricks.sdk import WorkspaceClient

wc = WorkspaceClient()

try:
    eq_conn = wc.connections.get(name="earthquake_conn")
except Exception as e:
    raise RuntimeError("Earthquake connection not found") from e

assert eq_conn.options["host"], "Connection host missing"
assert eq_conn.options["base_path"], "Connection base_path missing"

Extract the data as the json

In [0]:
import requests
import json
import datetime

run_date_str = dbutils.widgets.get("run_date")

if run_date_str:
    today = datetime.datetime.strptime(run_date_str, "%Y%m%d").date()
else:
    today = datetime.date.today()

# Monday of current week
current_monday = today - datetime.timedelta(days=today.weekday())
week_start = current_monday.strftime("%Y%m%d")

# Previous week window
start_time = current_monday - datetime.timedelta(days=7)
end_time = current_monday - datetime.timedelta(days=1)

start_time_str = start_time.strftime("%Y%m%d")
end_time_str = end_time.strftime("%Y%m%d")

start_time = start_time.strftime("%Y-%m-%d")
end_time = end_time.strftime("%Y-%m-%d")

query = f"query?format=geojson&starttime={start_time}&endtime={end_time}"

base_url = eq_conn.options['host']
base_path = eq_conn.options["base_path"]
url = f"{base_url}{base_path}{query}"

try:
    response = requests.get(url, timeout=30, headers={"Accept": "application/json"})
    response.raise_for_status()
    response_json = response.json()
except requests.exceptions.RequestException as e:
    raise RuntimeError(f"API request failed: {e}") from e
except ValueError as e:
    raise RuntimeError(f"Invalid JSON response: {e}") from e

directory = f"/Volumes/{volume_path}/summary/week_start={week_start}"
dbutils.fs.mkdirs(directory)

earthquake_volume_file = f"{directory}/earthquake_summary_{start_time_str}_{end_time_str}.json"

try:
    dbutils.fs.put(earthquake_volume_file, json.dumps(response_json), overwrite=True)
except Exception as e:
    raise RuntimeError(f"Failed to write data to volume: {e}") from e